# GeoChat — Region-Level Grounding and Multi-Turn Conversation

GeoChat's grounding capability lets you ask questions about **specific regions** of a satellite image by specifying bounding-box coordinates in the prompt. This is GeoChat's most distinctive feature compared to generic VLMs.

**Region prompt format:** Include `{<x_min><y_min><x_max><y_max>}` coordinates (normalized 0–1) in the question:
```
What is located in the region {<0.2><0.1><0.5><0.4>}?
```

**Multi-turn format:** Continue the conversation by appending previous turns:
```
USER: <image>\nQuestion 1 ASSISTANT: Answer 1 USER: Question 2 ASSISTANT:
```

## References
- GeoChat paper: https://arxiv.org/abs/2311.15826
- GeoChat GitHub: https://github.com/mbzuai-oryx/GeoChat

## 1. Load Model

In [ ]:
import sys
import os
import torch
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPImageProcessor, BitsAndBytesConfig

sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

model_id = "MBZUAI/GeoChat"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN or None
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float32, token=HF_TOKEN or None
    )

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, token=HF_TOKEN or None)
image_processor = CLIPImageProcessor.from_pretrained(model_id, token=HF_TOKEN or None)
model.eval()
print("GeoChat loaded.")

## 2. Helper Functions

In [ ]:
SYSTEM_PROMPT = (
    "A chat between a curious user and an artificial intelligence assistant "
    "specializing in remote sensing."
)


def ask_grounded(image_path, question, conversation_history=None, max_new_tokens=200):
    """Ask GeoChat a question, optionally continuing a conversation."""
    image = Image.open(image_path).convert("RGB")
    image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
    if torch.cuda.is_available():
        image_tensor = image_tensor.half().cuda()

    if conversation_history:
        history_str = " ".join(
            f"USER: {'<image>\\n' if i==0 else ''}{q} ASSISTANT: {a}"
            for i, (q, a) in enumerate(conversation_history)
        )
        prompt = f"{SYSTEM_PROMPT} {history_str} USER: {question} ASSISTANT:"
    else:
        prompt = f"{SYSTEM_PROMPT} USER: <image>\n{question} ASSISTANT:"

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs, images=image_tensor, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=1.0,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split("ASSISTANT:")[-1].strip()


def visualize_with_region(image_path, bbox_norm, title=""):
    """Show image with a bounding box overlay. bbox_norm = [x_min, y_min, x_max, y_max] in 0-1."""
    img = np.array(Image.open(image_path).convert("RGB"))
    h, w = img.shape[:2]
    x_min, y_min, x_max, y_max = [v * s for v, s in zip(bbox_norm, [w, h, w, h])]

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(img)
    rect = patches.Rectangle(
        (x_min, y_min), x_max - x_min, y_max - y_min,
        linewidth=2, edgecolor="red", facecolor="none",
    )
    ax.add_patch(rect)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Region-Level Grounding

In [ ]:
image_path = "sample_images/airport.jpg"

# Ask about a specific region (top-right quadrant)
bbox = [0.5, 0.0, 1.0, 0.5]  # [x_min, y_min, x_max, y_max] normalized
bbox_str = f"{{<{bbox[0]:.1f}><{bbox[1]:.1f}><{bbox[2]:.1f}><{bbox[3]:.1f}>}}"
question = f"What is visible in the region {bbox_str} of this image?"

visualize_with_region(image_path, bbox, title="Region of interest (red box)")

print(f"Q: {question}")
response = ask_grounded(image_path, question)
print(f"A: {response}")

## 4. Comparing Multiple Regions

In [ ]:
regions = [
    ([0.0, 0.0, 0.5, 0.5], "top-left quadrant"),
    ([0.5, 0.5, 1.0, 1.0], "bottom-right quadrant"),
    ([0.2, 0.3, 0.8, 0.7], "central region"),
]

print("Region comparison for airport image:")
print("=" * 55)

for bbox, region_name in regions:
    bbox_str = f"{{<{bbox[0]:.1f}><{bbox[1]:.1f}><{bbox[2]:.1f}><{bbox[3]:.1f}>}}"
    question = f"Describe what is in the region {bbox_str}."
    response = ask_grounded(image_path, question, max_new_tokens=80)
    print(f"\n[{region_name.upper()}]")
    print(f"  {response}")

## 5. Multi-Turn Remote Sensing Dialogue

In [ ]:
urban_image = "sample_images/urban.jpg"

# Turn 1
q1 = "Give me a high-level description of this aerial image."
a1 = ask_grounded(urban_image, q1, max_new_tokens=100)
print(f"Turn 1\nQ: {q1}")
print(f"A: {a1}")

history = [(q1, a1)]

# Turn 2 — drill down
q2 = "What type of urban environment is this? Dense or sparse?"
a2 = ask_grounded(urban_image, q2, conversation_history=history, max_new_tokens=80)
print(f"\nTurn 2\nQ: {q2}")
print(f"A: {a2}")

history.append((q2, a2))

# Turn 3 — infrastructure
q3 = "Can you identify any transportation infrastructure visible in this scene?"
a3 = ask_grounded(urban_image, q3, conversation_history=history, max_new_tokens=100)
print(f"\nTurn 3\nQ: {q3}")
print(f"A: {a3}")

## 6. Use Cases for GeoChat

| Use Case | Example Prompt |
|---|---|
| Rapid image triage | "Is there evidence of flooding in this image?" |
| Damage assessment | "Describe any visible damage or destruction in this scene." |
| Land cover description | "What percentage of this image is forested area?" |
| Infrastructure inventory | "List all transportation infrastructure visible." |
| Change detection (conceptual) | Given T1 + T2 images: "What changed between these two images?" |
| Region interrogation | "What is in the region `{<0.3><0.4><0.7><0.8>}`?" |

**Limitation**: GeoChat processes one image at a time and is not designed for pixel-precise tasks. For quantitative analysis (area measurement, change detection metrics), use task-specific models (Open-CD, samgeo, etc.).